# Visualise noise vectors: numerical vs. analytic hybridisation
Compares individual components of the three noise vectors (light inside the cavity, cavity output light and hybridised positions) computed by Numerical H (or LAM) and by the analytic methods, for the phase-dependent x-y systems (`PHASE_DEPENDENT_xy_dict`). Sections: one component near the positive resonances, the same component over the whole ± spectrum, a bar chart of the resonance peaks of every component (2×2, 3×2, 4×2), and every component of the 2×2 system over the whole spectrum.

Runtime: about 4-5 min (mostly the per-frequency loops on the 4×2 system and the ~80 figures of the last section).

# START

# 1 noise vector component
## around positive resonances

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *
# the PHASE_*_xy_dict fixtures are built lazily, so `from levisim import *` does not bring them in
from levisim.multiphysics.utils.shelf import PHASE_DEPENDENT_xy_dict, PHASE_EQUAL_xy_dict

system, ω_range = '3×2', np.arange(285, 310 + 0.005, 0.005) * kHz
system, ω_range = '4×2', np.arange(275, 310 + 0.005, 0.005) * kHz
system, ω_range = '2×2', np.arange(295, 310 + 0.005, 0.005) * kHz
mtrap = PHASE_EQUAL_xy_dict[system][+1]
mtrap = PHASE_EQUAL_xy_dict[system][-1]
mtrap = PHASE_DEPENDENT_xy_dict[system][+1]
mtrap = PHASE_DEPENDENT_xy_dict[system][-1]
mtrap.ω_range = ω_range

method_A = QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL
method_A = QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION

method_B = QuantumLinearTheoryMethod.ANALYTIC_CAVITY_OUTPUT
method_B = QuantumLinearTheoryMethod.ANALYTIC_INSIDE_CAVITY

noise_vector_type = NoiseVectorType.HYBRIDISED_POSITIONS
noise_vector_type = NoiseVectorType.CAVITY_OUTPUT_LIGHT
noise_vector_type = NoiseVectorType.LIGHT_INSIDE_CAVITY

match noise_vector_type:
    case NoiseVectorType.HYBRIDISED_POSITIONS:
        func = mtrap.hybridised_positions
        ylabel = r'Spectral position (zpf-scaled) [$\sqrt{\mathregular{phonons}/\mathregular{Hz}}$]'
    case NoiseVectorType.LIGHT_INSIDE_CAVITY:
        func = mtrap.light_noise_vectors
        ylabel = r'[$\sqrt{\mathregular{photons}/\mathregular{Hz}}$]'
    case NoiseVectorType.CAVITY_OUTPUT_LIGHT:
        func = mtrap.output_light_noise_vectors
        ylabel = r'[$\sqrt{\mathregular{photons}/\mathregular{Hz}}$]'

func_vals_A = np.array([func(method_A, ω) for ω in ω_range])
func_vals_B = np.array([func(method_B, ω) for ω in ω_range])

dim_index, noise_index = 1, mtrap.num_operators - 1
if noise_vector_type.is_light:
    assert dim_index <= 1
vals_A = func_vals_A[:, dim_index, noise_index]
vals_B = func_vals_B[:, dim_index, noise_index]

visualise_two_noise_component_calculation_methods(
    mtrap, noise_vector_type,
    dim_index, noise_index,
    vals_A, vals_B,
    method_A, method_B,
    ylabel=ylabel,
    plot_by_magnitude=True,
    return_resonances=True,
)

## over entire ± spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *
# the PHASE_*_xy_dict fixtures are built lazily, so `from levisim import *` does not bring them in
from levisim.multiphysics.utils.shelf import PHASE_DEPENDENT_xy_dict, PHASE_EQUAL_xy_dict

system, lim = '3×2', 285
system, lim = '4×2', 275
system, lim = '2×2', 295
mtrap = PHASE_EQUAL_xy_dict[system][+1]
mtrap = PHASE_EQUAL_xy_dict[system][-1]
mtrap = PHASE_DEPENDENT_xy_dict[system][+1]
mtrap = PHASE_DEPENDENT_xy_dict[system][-1]

method_A = QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL
method_A = QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION

method_B = QuantumLinearTheoryMethod.ANALYTIC_CAVITY_OUTPUT
method_B = QuantumLinearTheoryMethod.ANALYTIC_INSIDE_CAVITY

noise_vector_type = NoiseVectorType.HYBRIDISED_POSITIONS
noise_vector_type = NoiseVectorType.CAVITY_OUTPUT_LIGHT
noise_vector_type = NoiseVectorType.LIGHT_INSIDE_CAVITY

match noise_vector_type:
    case NoiseVectorType.HYBRIDISED_POSITIONS:
        func = mtrap.hybridised_positions
        ylabel = r'Spectral position (zpf-scaled) [$\sqrt{\mathregular{phonons}/\mathregular{Hz}}$]'
    case NoiseVectorType.LIGHT_INSIDE_CAVITY:
        func = mtrap.light_noise_vectors
        ylabel = r'[$\sqrt{\mathregular{photons}/\mathregular{Hz}}$]'
    case NoiseVectorType.CAVITY_OUTPUT_LIGHT:
        func = mtrap.output_light_noise_vectors
        ylabel = r'[$\sqrt{\mathregular{photons}/\mathregular{Hz}}$]'


dim_index, noise_index = 1, 4
HD_ω_range = np.arange(lim, 310 + 0.005, 0.005) * kHz
SD_ω_range = np.arange(-lim, lim + 1, 1) * kHz

ω_range = -HD_ω_range
mtrap.ω_range = ω_range
HD_minus_vals_A = np.array([func(method_A, ω) for ω in ω_range])
HD_minus_vals_B = np.array([func(method_B, ω) for ω in ω_range])

ω_range = SD_ω_range
mtrap.ω_range = ω_range
SD_vals_A = np.array([func(method_A, ω) for ω in ω_range])
SD_vals_B = np.array([func(method_B, ω) for ω in ω_range])

ω_range = HD_ω_range
mtrap.ω_range = ω_range
HD_plus_vals_A = np.array([func(method_A, ω) for ω in ω_range])
HD_plus_vals_B = np.array([func(method_B, ω) for ω in ω_range])

visualise_two_noise_component_calculation_methods_over_entire_spectrum(
    mtrap, noise_vector_type,
    HD_ω_range, SD_ω_range,
    HD_minus_vals_A, HD_minus_vals_B,
    SD_vals_A, SD_vals_B,
    HD_plus_vals_A, HD_plus_vals_B,
    dim_index, noise_index,
    method_A, method_B,
    ylabel=ylabel,
)
visualise_two_noise_component_calculation_methods_over_entire_spectrum(
    mtrap, noise_vector_type,
    HD_ω_range, SD_ω_range,
    HD_minus_vals_A, HD_minus_vals_B,
    SD_vals_A, SD_vals_B,
    HD_plus_vals_A, HD_plus_vals_B,
    dim_index, noise_index,
    method_A, method_B,
    ylabel=ylabel,
    log=True,
)

# All noise vectors and components

## visualise resonance peaks only

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *
# the PHASE_*_xy_dict fixtures are built lazily, so `from levisim import *` does not bring them in
from levisim.multiphysics.utils.shelf import PHASE_DEPENDENT_xy_dict, PHASE_EQUAL_xy_dict

all_peaks = {}
for system, lim in [
    ('2×2', 295),
    ('3×2', 285),
    ('4×2', 275),
]:
    all_peaks[system] = {}
    for mtrap in [
        # PHASE_EQUAL_xy_dict[system][+1],
        # PHASE_EQUAL_xy_dict[system][-1],
        # PHASE_DEPENDENT_xy_dict[system][+1],
        PHASE_DEPENDENT_xy_dict[system][-1],
    ]:
        ω_range = np.concatenate((
            np.arange(-310, -lim + 0.005, 0.005) * kHz,
            np.arange(-lim, lim + 1, 1) * kHz,
            np.arange(lim, 310 + 0.005, 0.005) * kHz,
        ))
        mtrap.ω_range = ω_range
        for method_A in [
            # QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL,
            QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION,
        ]:
            for method_B in [
                QuantumLinearTheoryMethod.ANALYTIC_INSIDE_CAVITY,
                # QuantumLinearTheoryMethod.ANALYTIC_CAVITY_OUTPUT,
            ]:
                for noise_vector_type in [
                    NoiseVectorType.LIGHT_INSIDE_CAVITY,
                    NoiseVectorType.CAVITY_OUTPUT_LIGHT,
                    NoiseVectorType.HYBRIDISED_POSITIONS,
                ]:
                    all_peaks[system][noise_vector_type] = {}

                    match noise_vector_type:
                        case NoiseVectorType.HYBRIDISED_POSITIONS:
                            func = mtrap.hybridised_positions
                        case NoiseVectorType.LIGHT_INSIDE_CAVITY:
                            func = mtrap.light_noise_vectors
                        case NoiseVectorType.CAVITY_OUTPUT_LIGHT:
                            func = mtrap.output_light_noise_vectors

                    func_vals_A = np.array([func(method_A, ω) for ω in ω_range])
                    func_vals_B = np.array([func(method_B, ω) for ω in ω_range])

                    N_dims, N_ops = mtrap.num_dimensions, mtrap.num_operators

                    # dim_slice = slice(0, 3)
                    dim_slice = slice(0, N_dims)
                    for dim_index in list(range(N_dims))[dim_slice]:

                        all_peaks[system][noise_vector_type][dim_index] = {}

                        # noise_slice = slice(0, 3)
                        noise_slice = slice(0, N_ops)
                        for noise_index in list(range(N_ops))[noise_slice]:

                            if noise_vector_type.is_light and dim_index > 1:
                                continue

                            vals_A = func_vals_A[:, dim_index, noise_index]
                            vals_B = func_vals_B[:, dim_index, noise_index]

                            peaks = visualise_two_noise_component_calculation_methods(
                                mtrap, noise_vector_type,
                                dim_index, noise_index,
                                vals_A, vals_B,
                                method_A, method_B,
                                visualise=False,
                                return_resonances=True,
                            )
                            name_str = noise_vector_string(mtrap, noise_vector_type, dim_index, noise_index)
                            all_peaks[system][noise_vector_type][dim_index][name_str] = peaks


# https://matplotlib.org/stable/gallery/lines_bars_and_markers/barchart.html
for system in [
    '2×2',
    '3×2',
    '4×2',
]:
    for noise_vector_type in [
        NoiseVectorType.LIGHT_INSIDE_CAVITY,
        NoiseVectorType.CAVITY_OUTPUT_LIGHT,
        NoiseVectorType.HYBRIDISED_POSITIONS,
    ]:
        mtrap = PHASE_DEPENDENT_xy_dict[system][+1]
        dim_range = range(2) if noise_vector_type.is_light else range(mtrap.num_dimensions)
        for dim_index in dim_range:
            method = QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION
            peak_data = all_peaks[system][noise_vector_type][dim_index]

            noise_components = list(peak_data.keys())
            # peak_bundles = np.array([_[QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION][:3] for _ in data.values()])
            peak_bundles = [_[method] for _ in peak_data.values()]
            x_range = np.arange(len(peak_bundles))

            max_num_peaks = max([len(_[method]) for _ in peak_data.values()])
            print(f'{max_num_peaks=}')

            # pad the bundles of peaks with zeros so that they all have the same length
            # https://stackoverflow.com/questions/45422000/add-n-zeros-to-the-end-of-an-array
            peak_bundle_arrays = []
            for peak_bundle in peak_bundles:
                N = max_num_peaks - len(peak_bundle)
                peak_bundle_array = np.array(peak_bundle)
                peak_bundle_array = np.pad(peak_bundle_array, (0, N), 'constant')
                peak_bundle_arrays.append(peak_bundle_array)
            peak_bundle_arrays = np.array(peak_bundle_arrays)

            fig, ax = plt.subplots(figsize=(LATEX_FIG_WIDTH, 4), dpi=150)
            ax.axhline(1, color=black, alpha=0.1)

            # the width of the bars
            width = 1 / (max_num_peaks + 1)
            multiplier = 0
            for ii in range(max_num_peaks):
                offset = width * multiplier
                rects = ax.bar(x_range + offset, peak_bundle_arrays[:, ii], width, label=f'peak {ii + 1}')
                multiplier += 1

            # Add some text for labels, title and custom x-axis tick labels, etc.
            operator_name = noise_components[0].split('[')[0]
            ax.set_title(f'Noise peaks for "{repr(noise_vector_type)}" ({operator_name=})')
            ax.set_xticks(x_range + (max_num_peaks / 2) * width, noise_components, rotation=45, ha='right', rotation_mode='anchor')
            ax.legend(loc='upper left', ncols=1 + max_num_peaks // 4, prop=dict(size=8))
            ax.set_yscale('log')
            ax.set_ylim(1e-9, 1e6)
            plt.show()

## plot over entire ± spectrum

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from levisim import *
# the PHASE_*_xy_dict fixtures are built lazily, so `from levisim import *` does not bring them in
from levisim.multiphysics.utils.shelf import PHASE_DEPENDENT_xy_dict, PHASE_EQUAL_xy_dict

for system, lim in [
    # ('4×2', 275),
    # ('3×2', 285),
    ('2×2', 295),
]:
    for mtrap in [
        # PHASE_EQUAL_xy_dict[system][+1],
        # PHASE_EQUAL_xy_dict[system][-1],
        # PHASE_DEPENDENT_xy_dict[system][+1],
        PHASE_DEPENDENT_xy_dict[system][-1],
    ]:
        for method_A in [
            # QuantumLinearTheoryMethod.LINEAR_AMPLIFIER_MODEL,
            QuantumLinearTheoryMethod.NUMERICAL_HYBRIDISATION,
        ]:
            for method_B in [
                QuantumLinearTheoryMethod.ANALYTIC_INSIDE_CAVITY,
                # QuantumLinearTheoryMethod.ANALYTIC_CAVITY_OUTPUT,
            ]:
                for noise_vector_type in [
                    NoiseVectorType.LIGHT_INSIDE_CAVITY,
                    NoiseVectorType.CAVITY_OUTPUT_LIGHT,
                    NoiseVectorType.HYBRIDISED_POSITIONS,
                ]:
                    match noise_vector_type:
                        case NoiseVectorType.HYBRIDISED_POSITIONS:
                            func = mtrap.hybridised_positions
                            ylabel = r'Spectral position [m$/\sqrt{\mathregular{Hz}}$]'
                        case NoiseVectorType.LIGHT_INSIDE_CAVITY:
                            func = mtrap.light_noise_vectors
                            ylabel = r'[$\sqrt{\mathregular{photons}/\mathregular{Hz}}$]'
                        case NoiseVectorType.CAVITY_OUTPUT_LIGHT:
                            func = mtrap.output_light_noise_vectors
                            ylabel = r'[$\sqrt{\mathregular{photons}/\mathregular{Hz}}$]'

                    HD_ω_range = np.arange(lim, 310 + 0.005, 0.005) * kHz
                    SD_ω_range = np.arange(-lim, lim + 1, 1) * kHz

                    ω_range = -HD_ω_range
                    mtrap.ω_range = ω_range
                    HD_minus_vals_A = np.array([func(method_A, ω) for ω in ω_range])
                    HD_minus_vals_B = np.array([func(method_B, ω) for ω in ω_range])

                    ω_range = SD_ω_range
                    mtrap.ω_range = ω_range
                    SD_vals_A = np.array([func(method_A, ω) for ω in ω_range])
                    SD_vals_B = np.array([func(method_B, ω) for ω in ω_range])

                    ω_range = HD_ω_range
                    mtrap.ω_range = ω_range
                    HD_plus_vals_A = np.array([func(method_A, ω) for ω in ω_range])
                    HD_plus_vals_B = np.array([func(method_B, ω) for ω in ω_range])

                    N_dims, N_ops = mtrap.num_dimensions, mtrap.num_operators
                    dim_slice = slice(0, 3)
                    dim_slice = slice(0, N_dims)
                    for dim_index in list(range(N_dims))[dim_slice]:
                        noise_slice = slice(0, 3)
                        noise_slice = slice(0, N_ops)
                        for noise_index in list(range(N_ops))[noise_slice]:
                            if noise_vector_type.is_light and dim_index > 1:
                                continue
                            visualise_two_noise_component_calculation_methods_over_entire_spectrum(
                                mtrap, noise_vector_type,
                                HD_ω_range, SD_ω_range,
                                HD_minus_vals_A, HD_minus_vals_B,
                                SD_vals_A, SD_vals_B,
                                HD_plus_vals_A, HD_plus_vals_B,
                                dim_index, noise_index,
                                method_A, method_B,
                                ylabel=ylabel,
                            )


# END